In [3]:
import numpy as np
import pandas as pd
import networkx as nx
from collections import Counter

import os

import matplotlib.pyplot as plt
import seaborn as sns

from namespaces import DA

os.chdir('/home/mai/notebooks/final_thesis/')
os.getcwd()

'/home/mai/notebooks/final_thesis'

In [5]:
data = np.load(os.path.join(DA.paths.data, 'dgraphfin.npz'))
print(f'Keys in .npz: {list(data.keys())}')

Keys in .npz: ['x', 'y', 'edge_index', 'edge_type', 'edge_timestamp', 'train_mask', 'valid_mask', 'test_mask']


In [6]:
x          = data['x']            # [N, feat_dim] node features
y          = data['y']            # [N] labels
edge_index = data['edge_index']   # [2, E] or [E, 2] — check shape
edge_type  = data.get('edge_type', None)
edge_timestamp = data.get('edge_timestamp', None)

In [7]:
# Handle edge_index shape — could be [2, E] or [E, 2]
if edge_index.shape[0] == 2:
    src = edge_index[0]
    dst = edge_index[1]
elif edge_index.shape[1] == 2:
    src = edge_index[:, 0]
    dst = edge_index[:, 1]
else:
    raise ValueError(f'Unexpected edge_index shape: {edge_index.shape}')

In [8]:
N = x.shape[0]
E = len(src)
 
print(f'\n{"="*60}')
print(f'BASIC STATISTICS')
print(f'{"="*60}')
print(f'Nodes:              {N:,}')
print(f'Edges:              {E:,}')
print(f'Feature dimensions: {x.shape[1]}')
print(f'Avg degree (dir.):  {E / N:.2f}')


BASIC STATISTICS
Nodes:              3,700,550
Edges:              4,300,999
Feature dimensions: 17
Avg degree (dir.):  1.16


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# Label distribution
# ─────────────────────────────────────────────────────────────────────────────
 
print(f'\n{"="*60}')
print(f'LABEL DISTRIBUTION')
print(f'{"="*60}')
 
label_counts = Counter(y)
for label, count in sorted(label_counts.items()):
    pct = 100 * count / N
    print(f'  Label {label:>3d}: {count:>10,}  ({pct:.2f}%)')


LABEL DISTRIBUTION
  Label   0:  1,210,092  (32.70%)
  Label   1:     15,509  (0.42%)
  Label   2:  1,620,851  (43.80%)
  Label   3:    854,098  (23.08%)


In [10]:
# Fraud-specific stats (label=1 is fraud in DGraphFin)
n_fraud     = int((y == 1).sum())
n_benign    = int((y == 0).sum())
n_background = N - n_fraud - n_benign
print(f'\n  Fraud (1):       {n_fraud:,}  ({100*n_fraud/N:.2f}%)')
print(f'  Benign (0):      {n_benign:,}  ({100*n_benign/N:.2f}%)')
print(f'  Background (2+): {n_background:,}  ({100*n_background/N:.2f}%)')


  Fraud (1):       15,509  (0.42%)
  Benign (0):      1,210,092  (32.70%)
  Background (2+): 2,474,949  (66.88%)


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# Edge type distribution (if available)
# ─────────────────────────────────────────────────────────────────────────────
 
if edge_type is not None:
    print(f'\n{"="*60}')
    print(f'EDGE TYPE DISTRIBUTION')
    print(f'{"="*60}')
    etype_counts = Counter(edge_type)
    for etype, count in sorted(etype_counts.items()):
        pct = 100 * count / E
        print(f'  Type {etype:>3d}: {count:>10,}  ({pct:.2f}%)')


EDGE TYPE DISTRIBUTION
  Type   1:    362,904  (8.44%)
  Type   2:     51,421  (1.20%)
  Type   3:    149,977  (3.49%)
  Type   4:    911,655  (21.20%)
  Type   5:  1,604,218  (37.30%)
  Type   6:    645,389  (15.01%)
  Type   7:     15,882  (0.37%)
  Type   8:     84,338  (1.96%)
  Type   9:    125,026  (2.91%)
  Type  10:    262,515  (6.10%)
  Type  11:     87,674  (2.04%)


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# Degree statistics (without NetworkX — faster for large graphs)
# ─────────────────────────────────────────────────────────────────────────────
 
print(f'\n{"="*60}')
print(f'DEGREE STATISTICS (directed)')
print(f'{"="*60}')
 
out_deg = np.bincount(src, minlength=N)
in_deg  = np.bincount(dst, minlength=N)
total_deg = out_deg + in_deg
 
for name, deg in [('Out-degree', out_deg), ('In-degree', in_deg), ('Total degree', total_deg)]:
    print(f'\n  {name}:')
    print(f'    Mean:   {deg.mean():.2f}')
    print(f'    Median: {np.median(deg):.0f}')
    print(f'    Std:    {deg.std():.2f}')
    print(f'    Min:    {deg.min()}')
    print(f'    Max:    {deg.max()}')
    print(f'    Isolated (deg=0): {(deg == 0).sum():,}')


DEGREE STATISTICS (directed)

  Out-degree:
    Mean:   1.16
    Median: 1
    Std:    1.29
    Min:    0
    Max:    6
    Isolated (deg=0): 1,509,888

  In-degree:
    Mean:   1.16
    Median: 1
    Std:    1.55
    Min:    0
    Max:    882
    Isolated (deg=0): 1,001,676

  Total degree:
    Mean:   2.32
    Median: 2
    Std:    1.91
    Min:    1
    Max:    882
    Isolated (deg=0): 0


In [13]:
# Degree by label
print(f'\n  Degree by label (total degree):')
for label in sorted(label_counts.keys()):
    mask = (y == label)
    deg_label = total_deg[mask]
    print(f'    Label {label}: mean={deg_label.mean():.2f}, median={np.median(deg_label):.0f}, std={deg_label.std():.2f}')


  Degree by label (total degree):
    Label 0: mean=2.89, median=2, std=1.92
    Label 1: mean=1.95, median=2, std=1.47
    Label 2: mean=1.97, median=2, std=1.94
    Label 3: mean=2.20, median=2, std=1.62


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Build NetworkX graph for structural statistics
# ─────────────────────────────────────────────────────────────────────────────
 
print(f'\n{"="*60}')
print(f'BUILDING NETWORKX GRAPH (this may take a moment)...')
print(f'{"="*60}')
 
G = nx.DiGraph()
G.add_nodes_from(range(N))
edges = list(zip(src.tolist(), dst.tolist()))
G.add_edges_from(edges)
 
print(f'  NetworkX nodes: {G.number_of_nodes():,}')
print(f'  NetworkX edges: {G.number_of_edges():,}')
print(f'  Self-loops:     {nx.number_of_selfloops(G):,}')
print(f'  Density:        {nx.density(G):.8f}')


BUILDING NETWORKX GRAPH (this may take a moment)...
  NetworkX nodes: 3,700,550
  NetworkX edges: 4,300,999
  Self-loops:     0
  Density:        0.00000031


In [ ]:
print(f'  Density:        {nx.density(G):.2e}')

  Density:        3.14e-07


In [21]:
nx.average_clustering(G)

0.03440537757402263

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# Connected components (on undirected view)
# ─────────────────────────────────────────────────────────────────────────────
 
print(f'\n{"="*60}')
print(f'CONNECTED COMPONENTS (undirected)')
print(f'{"="*60}')
 
G_undir = G.to_undirected()
components = list(nx.connected_components(G_undir))
component_sizes = sorted([len(c) for c in components], reverse=True)
 
print(f'  Number of components:        {len(components):,}')
print(f'  Largest component:           {component_sizes[0]:,} nodes ({100*component_sizes[0]/N:.2f}%)')
if len(component_sizes) > 1:
    print(f'  Second largest:              {component_sizes[1]:,} nodes')
print(f'  Isolated nodes (size=1):     {sum(1 for s in component_sizes if s == 1):,}')
print(f'  Component size distribution:')
size_counts = Counter(component_sizes)
for size, count in sorted(size_counts.items(), key=lambda x: -x[1])[:10]:
    print(f'    Size {size:>8,}: {count:,} components')
 
# Free undirected copy
del G_undir


CONNECTED COMPONENTS (undirected)
  Number of components:        1
  Largest component:           3,700,550 nodes (100.00%)
  Isolated nodes (size=1):     0
  Component size distribution:
    Size 3,700,550: 1 components


In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# Homophily / Heterophily ratio
# ─────────────────────────────────────────────────────────────────────────────
 
print(f'\n{"="*60}')
print(f'HOMOPHILY / HETEROPHILY ANALYSIS')
print(f'{"="*60}')
 
# Label categories:
#   0 = benign, 1 = fraud, 2/3 = background
# Map to readable names
label_names = {0: 'Benign', 1: 'Fraud'}
unique_labels = sorted(set(y))
for lbl in unique_labels:
    if lbl not in label_names:
        label_names[lbl] = f'Background({lbl})'
 
y_src = y[src]
y_dst = y[dst]
 
# ── Part 1: Labelled-only analysis (fraud & benign edges) ────────────────
print(f'\n  --- Labelled nodes only (fraud=1, benign=0) ---')
 
labelled_mask = (y_src <= 1) & (y_dst <= 1)
y_src_lab = y_src[labelled_mask]
y_dst_lab = y_dst[labelled_mask]
 
total_labelled_edges = int(labelled_mask.sum())
same_label  = int((y_src_lab == y_dst_lab).sum())
cross_label = total_labelled_edges - same_label
 
ff = int(((y_src_lab == 1) & (y_dst_lab == 1)).sum())
fb = int(((y_src_lab == 1) & (y_dst_lab == 0)).sum())
bf = int(((y_src_lab == 0) & (y_dst_lab == 1)).sum())
bb = int(((y_src_lab == 0) & (y_dst_lab == 0)).sum())
 
if total_labelled_edges > 0:
    homophily = same_label / total_labelled_edges
    print(f'  Labelled edges:        {total_labelled_edges:,}')
    print(f'  Same-label edges:      {same_label:,} ({100*same_label/total_labelled_edges:.2f}%)')
    print(f'  Cross-label edges:     {cross_label:,} ({100*cross_label/total_labelled_edges:.2f}%)')
    print(f'  Edge homophily ratio:  {homophily:.4f}')
    print(f'  Edge heterophily ratio: {1 - homophily:.4f}')
    print(f'\n  Edge breakdown (labelled only):')
    print(f'    Fraud  → Fraud:   {ff:>10,}  ({100*ff/total_labelled_edges:.2f}%)')
    print(f'    Fraud  → Benign:  {fb:>10,}  ({100*fb/total_labelled_edges:.2f}%)')
    print(f'    Benign → Fraud:   {bf:>10,}  ({100*bf/total_labelled_edges:.2f}%)')
    print(f'    Benign → Benign:  {bb:>10,}  ({100*bb/total_labelled_edges:.2f}%)')
 
    fraud_edges  = ff + fb
    benign_edges = bb + bf
    if fraud_edges > 0:
        print(f'\n  Fraud node homophily:  {ff/fraud_edges:.4f} ({ff:,}/{fraud_edges:,})')
    if benign_edges > 0:
        print(f'  Benign node homophily: {bb/benign_edges:.4f} ({bb:,}/{benign_edges:,})')
 
# ── Part 2: Full analysis including background nodes ─────────────────────
print(f'\n  --- All edges including background nodes ---')
 
# Full edge type matrix: count edges for every (src_label, dst_label) pair
print(f'\n  Edge matrix (src_label → dst_label):')
print(f'  {"":>15s}', end='')
for dl in unique_labels:
    print(f'  {label_names[dl]:>15s}', end='')
print()
 
edge_matrix = {}
for sl in unique_labels:
    print(f'  {label_names[sl]:>15s}', end='')
    for dl in unique_labels:
        count = int(((y_src == sl) & (y_dst == dl)).sum())
        edge_matrix[(sl, dl)] = count
        print(f'  {count:>15,}', end='')
    print()
 
total_edges = sum(edge_matrix.values())
print(f'\n  Total edges: {total_edges:,}')
 
# Overall homophily across all label types
same_all  = sum(edge_matrix[(l, l)] for l in unique_labels)
cross_all = total_edges - same_all
print(f'  Same-label (all):     {same_all:,} ({100*same_all/total_edges:.2f}%)')
print(f'  Cross-label (all):    {cross_all:,} ({100*cross_all/total_edges:.2f}%)')
print(f'  Homophily ratio (all): {same_all/total_edges:.4f}')
 
# ── Part 3: Per-class neighbourhood composition ─────────────────────────
# For each class, what fraction of its outgoing edges go to each other class
print(f'\n  Per-class outgoing edge distribution:')
for sl in unique_labels:
    row_total = sum(edge_matrix[(sl, dl)] for dl in unique_labels)
    if row_total == 0:
        continue
    print(f'\n    {label_names[sl]} nodes ({row_total:,} outgoing edges):')
    for dl in unique_labels:
        count = edge_matrix[(sl, dl)]
        pct = 100 * count / row_total
        print(f'      → {label_names[dl]:>15s}: {count:>10,}  ({pct:.2f}%)')
 
# ── Part 4: Per-class incoming edge distribution ─────────────────────────
print(f'\n  Per-class incoming edge distribution:')
for dl in unique_labels:
    col_total = sum(edge_matrix[(sl, dl)] for sl in unique_labels)
    if col_total == 0:
        continue
    print(f'\n    {label_names[dl]} nodes ({col_total:,} incoming edges):')
    for sl in unique_labels:
        count = edge_matrix[(sl, dl)]
        pct = 100 * count / col_total
        print(f'      ← {label_names[sl]:>15s}: {count:>10,}  ({pct:.2f}%)')
 
# ── Part 5: Neighbourhood composition (undirected — all edges) ───────────
# For each target class, count how many edges (in either direction) connect
# to each neighbour class. An edge (u→v) contributes to both u's and v's
# neighbourhood, so we sum outgoing and incoming counts per class.
print(f'\n  --- Neighbourhood composition (undirected, all edges) ---')
 
for target_label in [1, 0]:   # fraud first, then benign
    # edges where target class is src + edges where target class is dst
    # but subtract self-edges (target→target) once to avoid double-counting
    neighbour_counts = {}
    for nl in unique_labels:
        if nl == target_label:
            # same-class: outgoing + incoming + internal (but internal = edge_matrix[(t,t)]
            # counted once as src and once as dst, so just add both)
            count = edge_matrix[(target_label, nl)] + edge_matrix[(nl, target_label)]
        else:
            count = edge_matrix[(target_label, nl)] + edge_matrix[(nl, target_label)]
        neighbour_counts[nl] = count
 
    total = sum(neighbour_counts.values())
    if total == 0:
        continue
 
    same_class = neighbour_counts[target_label]
    homophily_ratio = same_class / total
 
    print(f'\n    {label_names[target_label]} nodes ({total:,} total edges, undirected):')
    for nl in unique_labels:
        count = neighbour_counts[nl]
        pct = 100 * count / total
        print(f'      ↔ {label_names[nl]:>15s}: {count:>10,}  ({pct:.2f}%)')
    print(f'      Homophily ratio:   {homophily_ratio:.4f}')
    print(f'      Heterophily ratio: {1 - homophily_ratio:.4f}')
 
print(f'\n{"="*60}')
print(f'DONE')
print(f'{"="*60}')


HOMOPHILY / HETEROPHILY ANALYSIS

  --- Labelled nodes only (fraud=1, benign=0) ---
  Labelled edges:        746,271
  Same-label edges:      736,420 (98.68%)
  Cross-label edges:     9,851 (1.32%)
  Edge homophily ratio:  0.9868
  Edge heterophily ratio: 0.0132

  Edge breakdown (labelled only):
    Fraud  → Fraud:          249  (0.03%)
    Fraud  → Benign:       2,789  (0.37%)
    Benign → Fraud:        7,062  (0.95%)
    Benign → Benign:     736,171  (98.65%)

  Fraud node homophily:  0.0820 (249/3,038)
  Benign node homophily: 0.9905 (736,171/743,233)

  --- All edges including background nodes ---

  Edge matrix (src_label → dst_label):
                            Benign            Fraud    Background(2)    Background(3)
           Benign          736,171            7,062          864,466          483,524
            Fraud            2,789              249            5,860            2,690
    Background(2)          392,064            7,321          626,322          290,568
    B